# Inspect learned connection in AKOrN

This notebook tries to investigate the hidden patterns of learned AKOrN models. Specifically, this inspects $J_{ij}$, the connectivity matrices.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Add source directory to path
#sys.path.append('/source')
from source.models.classification.my_knet import MyAKOrN

from source.models.classification.analysis_utils import AKOrNStaticAnalyzer

from source.data.augs import augmentation_strong

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Define the sweep directories
results_dir = Path("results")
sweep_dirs = [
    "sweep_20250708_581384.opbs_0",
    "sweep_20250708_581385.opbs_1",
    "sweep_20250708_581386.opbs_2",
    "sweep_20250708_581387.opbs_3",
    "sweep_20250708_581388.opbs_4",
    "sweep_20250708_581389.opbs_5",
    "sweep_20250708_581390.opbs_6",
    "sweep_20250708_581391.opbs_7",
    "sweep_20250708_581392.opbs_8",
    "sweep_20250708_581393.opbs_9",
    "sweep_20250708_581394.opbs_10",
    "sweep_20250708_581395.opbs_11",
    "sweep_20250708_581396.opbs_12",
    "sweep_20250708_581397.opbs_13",
    "sweep_20250708_581398.opbs_14",
    "sweep_20250708_581399.opbs_15",
    "sweep_20250708_581400.opbs_16",
    "sweep_20250708_581401.opbs_17"
]

print(f"Found {len(sweep_dirs)} sweep directories")

## 1. Load Learned Model and Configuration

In [ ]:
# Load the best model checkpoint
checkpoint_path = "results/20250704_570979.opbs/my_akorn_cifar10_final.pth"
config_path = "results/20250704_570979.opbs/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in checkpoint_path:
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in checkpoint_path:
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
ModelAnalysis = AKOrNStaticAnalyzer(model)

In [ ]:
ModelAnalysis.show_full_basic_plots(0)

In [ ]:
ModelAnalysis.show_full_basic_plots(1)

In [ ]:
ModelAnalysis.show_full_basic_plots(2)